In [1]:
pip install ipynb

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import tkinter as tk
from tkinter import messagebox
from ipynb.fs.full.data_manager import initialise_file, save_job, get_employer_jobs, get_job_applicants


initialise_file()

root = tk.Tk()
root.title("WorkLink - Employer System")
root.geometry("750x650")
root.config(bg="#f0f0f0")

current_employer_id = "1" 
current_employer_name = "SamuelTech"

def clear_screen():
    for widget in root.winfo_children():
        widget.destroy()

def employer_dashboard():
    clear_screen()

    title = tk.Label(root, text="Employer Dashboard", font=("Arial", 22, "bold"), bg="#f0f0f0", fg="black")
    title.pack(pady=25)

    welcome = tk.Label(root, text=f"Welcome back, {current_employer_name}", font=("Arial", 14), bg="#f0f0f0", fg="black")
    welcome.pack(pady=10)

    post_btn = tk.Button(root, text="Post a New Job", font=("Arial", 12, "bold"), bg="green", fg="white", width=22, command=post_job_screen)
    post_btn.pack(pady=12)

    view_btn = tk.Button(root, text="View Posted Jobs", font=("Arial", 12, "bold"), bg="blue", fg="white", width=22, command=view_jobs_screen)
    view_btn.pack(pady=12)

    exit_btn = tk.Button(root, text="Exit Application", font=("Arial", 12, "bold"), bg="red", fg="white", width=22, command=root.destroy)
    exit_btn.pack(pady=12)

def post_job_screen():
    clear_screen()

    title = tk.Label(root, text="Post a New Job", font=("Arial", 20, "bold"), bg="#f0f0f0", fg="black")
    title.pack(pady=20)

    tk.Label(root, text="Job Title", font=("Arial", 11), bg="#f0f0f0", fg="black").pack(anchor="w", padx=150)
    job_title_entry = tk.Entry(root, width=50, font=("Arial", 11), relief="solid", bd=1)
    job_title_entry.pack(pady=5)

    tk.Label(root, text="Job Description", font=("Arial", 11), bg="#f0f0f0", fg="black").pack(anchor="w", padx=150)
    description_entry = tk.Text(root, width=50, height=5, font=("Arial", 11), bg="white", fg="black", relief="solid", bd=1)
    description_entry.pack(pady=5)

    tk.Label(root, text="Required Skills (e.g. Python, Excel)", font=("Arial", 11), bg="#f0f0f0", fg="black").pack(anchor="w", padx=150)
    skills_entry = tk.Entry(root, width=50, font=("Arial", 11), relief="solid", bd=1)
    skills_entry.pack(pady=5)

    tk.Label(root, text="Minimum Experience Level Required", font=("Arial", 11), bg="#f0f0f0", fg="black").pack(anchor="w", padx=150)
    experience_var = tk.StringVar()
    experience_var.set("Entry-Level")

    experience_menu = tk.OptionMenu(root, experience_var, "No Experience", "Entry-Level", "Intermediate", "Expert")
    experience_menu.config(font=("Arial", 11), bg="white", fg="black", width=15)
    experience_menu.pack(pady=8)

    def handle_post():
        title_text = job_title_entry.get().strip()
        desc_text = description_entry.get("1.0", tk.END).strip()
        skills_text = skills_entry.get().strip()
        exp_text = experience_var.get()

        if not title_text or not desc_text or not skills_text:
            messagebox.showerror("Validation Error", "All creation fields must be filled out.")
            return

        save_job(current_employer_id, title_text, desc_text, skills_text, exp_text)
        messagebox.showinfo("Success", "Job vacancy listed successfully!")
        employer_dashboard()

    save_btn = tk.Button(root, text="Publish Listing", font=("Arial", 12, "bold"), bg="green", fg="white", width=20, command=handle_post)
    save_btn.pack(pady=15)

    back_btn = tk.Button(root, text="Cancel & Back", font=("Arial", 12, "bold"), bg="gray", fg="white", width=20, command=employer_dashboard)
    back_btn.pack()

def view_jobs_screen():
    clear_screen()

    title = tk.Label(root, text="Your Job Vacancies", font=("Arial", 20, "bold"), bg="#f0f0f0", fg="black")
    title.pack(pady=15)

    canvas = tk.Canvas(root, bg="#f0f0f0", highlightthickness=0)
    scrollbar = tk.Scrollbar(root, orient="vertical", command=canvas.yview)
    scroll_frame = tk.Frame(canvas, bg="#f0f0f0")

    scroll_frame.bind(
        "<Configure>",
        lambda e: canvas.configure(scrollregion=canvas.bbox("all"))
    )
    canvas.create_window((0, 0), window=scroll_frame, anchor="nw", width=700)
    canvas.configure(yscrollcommand=scrollbar.set)

    canvas.pack(side="left", fill="both", expand=True, padx=20)
    scrollbar.pack(side="right", fill="y")

    my_posted_jobs = get_employer_jobs(current_employer_id)

    if not my_posted_jobs:
        tk.Label(scroll_frame, text="You have not published any job posts yet.", font=("Arial", 14, "bold"), bg="#f0f0f0", fg="gray").pack(pady=40)
    else:
        for job in my_posted_jobs:
            job_id = job["id"]

            job_box = tk.Frame(scroll_frame, bg="white", bd=1, relief="solid")
            job_box.pack(fill="x", padx=10, pady=10, ipady=5)

            tk.Label(job_box, text=job["title"], font=("Arial", 14, "bold"), bg="white", fg="blue").pack(anchor="w", padx=15, pady=5)
            tk.Label(job_box, text=f"Experience: {job['experience']}", font=("Arial", 11), bg="white", fg="black").pack(anchor="w", padx=15)
            tk.Label(job_box, text=f"Requirements: {job['skills']}", font=("Arial", 11), bg="white", fg="black").pack(anchor="w", padx=15)
            tk.Label(job_box, text=f"Description: {job['description']}", font=("Arial", 11), bg="white", fg="gray", wraplength=600, justify="left").pack(anchor="w", padx=15, pady=5)

            view_app_btn = tk.Button(
                job_box, 
                text="Review Applicants", 
                font=("Arial", 10, "bold"),
                bg="blue",
                fg="white",
                command=lambda j_id=job_id, j_title=job["title"]: view_applicants_screen(j_id, j_title)
            )
            view_app_btn.pack(anchor="e", padx=15, pady=5)

    back_btn = tk.Button(root, text="Return to Dashboard", font=("Arial", 12, "bold"), bg="gray", fg="white", command=employer_dashboard)
    back_btn.pack(side="bottom", pady=15)

def view_applicants_screen(job_id, job_title):
    clear_screen()

    title = tk.Label(root, text=f"Applicants for: {job_title}", font=("Arial", 18, "bold"), bg="#f0f0f0", fg="black")
    title.pack(pady=20)

    applicants = get_job_applicants(job_id)

    if not applicants:
        tk.Label(root, text="No candidates have applied for this vacancy yet.", font=("Arial", 14, "bold"), bg="#f0f0f0", fg="gray").pack(pady=50)
    else:
        for candidate in applicants:
            candidate_box = tk.Frame(root, bg="white", bd=1, relief="solid")
            candidate_box.pack(fill="x", padx=40, pady=8, ipady=6)

            tk.Label(candidate_box, text=candidate["name"], font=("Arial", 13, "bold"), bg="white", fg="black").pack(anchor="w", padx=15, pady=2)
            tk.Label(candidate_box, text=f"Contact Email: {candidate['email']}", font=("Arial", 11), bg="white", fg="gray").pack(anchor="w", padx=15)
            tk.Label(candidate_box, text=f"Candidate Skills: {candidate['skills']}", font=("Arial", 11), bg="white", fg="black").pack(anchor="w", padx=15, pady=2)

    back_btn = tk.Button(root, text="Back to Job Listings", font=("Arial", 12, "bold"), bg="gray", fg="white", command=view_jobs_screen)
    back_btn.pack(side="bottom", pady=25)

employer_dashboard()
root.mainloop()